
# 04 — Aberje Synthetic Audiences MVP Completo

Este notebook está completo para rodar o pipeline Aberje:

```text
df_agents + df_news
↓
1 chamada LLM por agente × notícia
↓
retorna todos os itens do questionário em JSON estruturado
↓
ThreadPoolExecutor
↓
gpt-4o-mini
↓
REPEATS = 1
↓
agregação de constructos
```

O notebook inclui:

1. leitura da base de agentes;
2. criação ou leitura do dataframe de notícias;
3. contexto completo do agente;
4. questionário completo 1–7;
5. structured output com Pydantic;
6. LangChain `ChatOpenAI`;
7. execução paralela;
8. checkpoints;
9. cálculo de constructos;
10. agregações para análise e apresentação.


## 1. Instalação e imports

In [1]:

# Rode uma vez, se necessário:
# !pip install -U pandas numpy pydantic langchain langchain-openai tqdm

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import time
import json
import pandas as pd
import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 220)


## 2. Configurações centrais

In [2]:

# Ajuste para a raiz do seu projeto local
PROJECT_DIR = Path(r"C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm")

DATA_OUTPUT_DIR = PROJECT_DIR / "data" / "output"
DATA_INPUT_DIR = PROJECT_DIR / "data" / "input"
ABERJE_OUTPUT_DIR = PROJECT_DIR / "data" / "aberje_outputs"

DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_INPUT_DIR.mkdir(parents=True, exist_ok=True)
ABERJE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "gpt-4o-mini"
TEMPERATURE = 0.2
DEFAULT_REPEATS = 1

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_OUTPUT_DIR:", DATA_OUTPUT_DIR)
print("DATA_INPUT_DIR:", DATA_INPUT_DIR)
print("ABERJE_OUTPUT_DIR:", ABERJE_OUTPUT_DIR)
print("MODEL:", MODEL)


PROJECT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm
DATA_OUTPUT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output
DATA_INPUT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\input
ABERJE_OUTPUT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\aberje_outputs
MODEL: gpt-4o-mini


## 3. Colunas esperadas

In [24]:

AGENT_ID_COLUMN = "id_persona"

PERSONA_COLUMNS = [
    "Persona",
]

SOCIO_DEMOGRAPHIC_COLUMNS = [
    "idade",
    "gênero",
    "escolaridade",
    "situação ocupacional",
    "estado civil",
    "condição de moradia",
    "tipo de moradia",
    "localização",
    "renda familiar mensal",
    "acesso à internet",
    "tamanho do domicílio",
    "telefone",
]

VALUES_ATTITUDES_PERCEPTIONS_COLUMNS = [
    "identificação política",
    "ideologia",
    "raça/cor",
    "geração",
    "segmento_e_bolhas",
    "tipo_de_território",
    "uf",
    "tipo_de_fontes_de_informação",
    "Religiosidade",
    "Família",
    "Honestidade",
    "Meritocracia",
    "Ideologia",
    "Discriminação",
    "Sexualidade",
    "Comportamento_social",
    "Identidade_nacional",
    "Insegurança",
    "Confiança",
]

AGENT_CONTEXT_COLUMNS = (
    [AGENT_ID_COLUMN]
    + PERSONA_COLUMNS
    + SOCIO_DEMOGRAPHIC_COLUMNS
    + VALUES_ATTITUDES_PERCEPTIONS_COLUMNS
)

NEWS_REQUIRED_COLUMNS = [
    "news_id",
    "titulo",
    "conteudo",
    "tipo_de_fontes_de_informação",
]

ITEM_COLUMNS = [
    "VAL_01", "VAL_02", "VAL_03", "VAL_04",
    "ARO_01", "ARO_02", "ARO_03",
    "POS_01", "POS_02", "POS_03", "POS_04", "POS_05",
    "NEG_01", "NEG_02", "NEG_03", "NEG_04", "NEG_05",
    "CRE_01", "CRE_02", "CRE_03",
    "ATT_01", "ATT_02", "ATT_03", "ATT_04",
]

CONSTRUCT_MAP = {
    "Valencia": ["VAL_01", "VAL_02", "VAL_03", "VAL_04"],
    "Arousal": ["ARO_01", "ARO_02", "ARO_03"],
    "Emocao_Positiva": ["POS_01", "POS_02", "POS_03", "POS_04", "POS_05"],
    "Emocao_Negativa": ["NEG_01", "NEG_02", "NEG_03", "NEG_04", "NEG_05"],
    "Credibilidade": ["CRE_01", "CRE_02", "CRE_03"],
    "Atratividade": ["ATT_01", "ATT_02", "ATT_03", "ATT_04"],
}


ITEM_COLUMNS = [
    "VAL_01", "VAL_02", "VAL_03", "VAL_04",
    "POS_01", "POS_02", "POS_03", "POS_04", "POS_05",
    "CRE_01", "CRE_02", "CRE_03",
]

CONSTRUCT_MAP = {
    "Valencia": ["VAL_01", "VAL_02", "VAL_03", "VAL_04"],
    "Emocao_Positiva": ["POS_01", "POS_02", "POS_03", "POS_04", "POS_05"],
    "Credibilidade": ["CRE_01", "CRE_02", "CRE_03"],
}

## 4. Schema de resposta estruturada

In [22]:

class SurveyResponse(BaseModel):
    # VALÊNCIA — 1 = polo negativo; 7 = polo positivo
    VAL_01: int = Field(ge=1, le=7, description="feliz-infeliz; 1=infeliz, 7=feliz")
    VAL_02: int = Field(ge=1, le=7, description="agradado-desagradado; 1=desagradado, 7=agradado")
    VAL_03: int = Field(ge=1, le=7, description="satisfeito-insatisfeito; 1=insatisfeito, 7=satisfeito")
    VAL_04: int = Field(ge=1, le=7, description="contente-melancolico; 1=melancolico, 7=contente")

    # AROUSAL — 1 = baixo arousal; 7 = alto arousal
    ARO_01: int = Field(ge=1, le=7, description="estimulado-relaxado; 1=relaxado, 7=estimulado")
    ARO_02: int = Field(ge=1, le=7, description="excitado-calmo; 1=calmo, 7=excitado")
    ARO_03: int = Field(ge=1, le=7, description="desperto-sonolento; 1=sonolento, 7=desperto")

    # EMOÇÃO POSITIVA — 1 = nada; 7 = extremamente
    POS_01: int = Field(ge=1, le=7, description="alerta; 1=nada, 7=extremamente")
    POS_02: int = Field(ge=1, le=7, description="inspirado; 1=nada, 7=extremamente")
    POS_03: int = Field(ge=1, le=7, description="determinado; 1=nada, 7=extremamente")
    POS_04: int = Field(ge=1, le=7, description="atento; 1=nada, 7=extremamente")
    POS_05: int = Field(ge=1, le=7, description="ativo; 1=nada, 7=extremamente")

    # EMOÇÃO NEGATIVA — 1 = nada; 7 = extremamente
    NEG_01: int = Field(ge=1, le=7, description="chateado; 1=nada, 7=extremamente")
    NEG_02: int = Field(ge=1, le=7, description="hostil; 1=nada, 7=extremamente")
    NEG_03: int = Field(ge=1, le=7, description="envergonhado; 1=nada, 7=extremamente")
    NEG_04: int = Field(ge=1, le=7, description="nervoso; 1=nada, 7=extremamente")
    NEG_05: int = Field(ge=1, le=7, description="com medo; 1=nada, 7=extremamente")

    # CREDIBILIDADE — 1 = descreve muito mal; 7 = descreve muito bem
    CRE_01: int = Field(ge=1, le=7, description="preciso; 1=descreve muito mal, 7=descreve muito bem")
    CRE_02: int = Field(ge=1, le=7, description="autentico; 1=descreve muito mal, 7=descreve muito bem")
    CRE_03: int = Field(ge=1, le=7, description="crivel; 1=descreve muito mal, 7=descreve muito bem")

    # ATRATIVIDADE — 1 = baixa atratividade; 7 = alta atratividade
    ATT_01: int = Field(ge=1, le=7, description="atraente-pouco atraente; 1=pouco atraente, 7=atraente")
    ATT_02: int = Field(ge=1, le=7, description="elegante-sem elegancia; 1=sem elegancia, 7=elegante")
    ATT_03: int = Field(ge=1, le=7, description="bonito-feio; 1=feio, 7=bonito")
    ATT_04: int = Field(ge=1, le=7, description="tem classe-nao tem classe; 1=nao tem classe, 7=tem classe")


class SurveyResponse(BaseModel):
    # VALÊNCIA
    VAL_01: int = Field(ge=1, le=7)
    VAL_02: int = Field(ge=1, le=7)
    VAL_03: int = Field(ge=1, le=7)
    VAL_04: int = Field(ge=1, le=7)

    # EMOÇÃO POSITIVA
    POS_01: int = Field(ge=1, le=7)
    POS_02: int = Field(ge=1, le=7)
    POS_03: int = Field(ge=1, le=7)
    POS_04: int = Field(ge=1, le=7)
    POS_05: int = Field(ge=1, le=7)

    # CREDIBILIDADE
    CRE_01: int = Field(ge=1, le=7)
    CRE_02: int = Field(ge=1, le=7)
    CRE_03: int = Field(ge=1, le=7)


## 5. Questionário completo

In [23]:

def build_questionnaire_text() -> str:
    return """
QUESTIONÁRIO COMPLETO

Responda todos os itens usando apenas números inteiros de 1 a 7.

VALÊNCIA
Instrução: Após ler o conteúdo acima, minha reação foi...
Para cada item, 1 representa o polo negativo e 7 representa o polo positivo.

VAL_01: 1 = infeliz | 7 = feliz
VAL_02: 1 = desagradado(a) | 7 = agradado(a)
VAL_03: 1 = insatisfeito(a) | 7 = satisfeito(a)
VAL_04: 1 = melancólico(a) | 7 = contente

AROUSAL
Instrução: Após ler o conteúdo acima, senti-me...
Para cada item, 1 representa menor ativação e 7 representa maior ativação.

ARO_01: 1 = relaxado(a) | 7 = estimulado(a)
ARO_02: 1 = calmo(a) | 7 = excitado(a)
ARO_03: 1 = sonolento(a) | 7 = desperto(a)

EMOÇÃO POSITIVA
Instrução: Ao ler o conteúdo acima, senti-me...
Escala: 1 = nada | 4 = moderadamente | 7 = extremamente.

POS_01: alerta
POS_02: inspirado(a)
POS_03: determinado(a)
POS_04: atento(a)
POS_05: ativo(a)

EMOÇÃO NEGATIVA
Instrução: Ao ler o conteúdo acima, senti-me...
Escala: 1 = nada | 4 = moderadamente | 7 = extremamente.

NEG_01: chateado(a)
NEG_02: hostil
NEG_03: envergonhado(a)
NEG_04: nervoso(a)
NEG_05: com medo

CREDIBILIDADE DA MENSAGEM
Instrução: O quanto cada adjetivo descreve o conteúdo acima?
Escala: 1 = descreve muito mal | 4 = intermediário | 7 = descreve muito bem.

CRE_01: preciso
CRE_02: autêntico
CRE_03: crível

ATRATIVIDADE DA FONTE
Instrução: A pessoa que aparece, assina ou representa o conteúdo acima me parece...
Caso não haja pessoa identificável, responda pensando na fonte, porta-voz, instituição ou assinatura do conteúdo.
Escala: 1 representa baixa atratividade; 7 representa alta atratividade.

ATT_01: 1 = pouco atraente | 7 = atraente
ATT_02: 1 = sem elegância | 7 = elegante
ATT_03: 1 = feio(a) | 7 = bonito(a)
ATT_04: 1 = não tem classe | 7 = tem classe
""".strip()




def build_questionnaire_text() -> str:
    return """
QUESTIONÁRIO

Responda todos os itens usando apenas números inteiros de 1 a 7.

VALÊNCIA
Após ler o conteúdo acima, minha reação foi...

VAL_01: 1 = infeliz | 7 = feliz
VAL_02: 1 = desagradado(a) | 7 = agradado(a)
VAL_03: 1 = insatisfeito(a) | 7 = satisfeito(a)
VAL_04: 1 = melancólico(a) | 7 = contente

EMOÇÃO POSITIVA
Ao ler o conteúdo acima, senti-me...
Escala: 1 = nada | 4 = moderadamente | 7 = extremamente.

POS_01: alerta
POS_02: inspirado(a)
POS_03: determinado(a)
POS_04: atento(a)
POS_05: ativo(a)

CREDIBILIDADE DA MENSAGEM
O quanto cada adjetivo descreve o conteúdo acima?
Escala: 1 = descreve muito mal | 4 = intermediário | 7 = descreve muito bem.

CRE_01: preciso
CRE_02: autêntico
CRE_03: crível
""".strip()

print(build_questionnaire_text())


QUESTIONÁRIO

Responda todos os itens usando apenas números inteiros de 1 a 7.

VALÊNCIA
Após ler o conteúdo acima, minha reação foi...

VAL_01: 1 = infeliz | 7 = feliz
VAL_02: 1 = desagradado(a) | 7 = agradado(a)
VAL_03: 1 = insatisfeito(a) | 7 = satisfeito(a)
VAL_04: 1 = melancólico(a) | 7 = contente

EMOÇÃO POSITIVA
Ao ler o conteúdo acima, senti-me...
Escala: 1 = nada | 4 = moderadamente | 7 = extremamente.

POS_01: alerta
POS_02: inspirado(a)
POS_03: determinado(a)
POS_04: atento(a)
POS_05: ativo(a)

CREDIBILIDADE DA MENSAGEM
O quanto cada adjetivo descreve o conteúdo acima?
Escala: 1 = descreve muito mal | 4 = intermediário | 7 = descreve muito bem.

CRE_01: preciso
CRE_02: autêntico
CRE_03: crível


## 6. Funções de validação e utilidades

In [25]:

def safe_get(row: pd.Series, column: str, default: str = "Não informado") -> str:
    if column not in row.index:
        return default
    value = row[column]
    if pd.isna(value):
        return default
    return str(value)


def validate_columns(df: pd.DataFrame, required_columns: list[str], name: str, raise_error: bool = True) -> list[str]:
    missing = [c for c in required_columns if c not in df.columns]
    if missing and raise_error:
        raise ValueError(f"{name} não contém todas as colunas esperadas. Colunas ausentes: {missing}")
    return missing


def format_column_block(row: pd.Series, columns: list[str]) -> str:
    return "\n".join(f"- {col}: {safe_get(row, col)}" for col in columns)


## 7. Contexto completo do agente

In [26]:

def build_agent_context(agent_row: pd.Series) -> str:
    persona = safe_get(agent_row, "Persona")
    socio = format_column_block(agent_row, SOCIO_DEMOGRAPHIC_COLUMNS)
    values = format_column_block(agent_row, VALUES_ATTITUDES_PERCEPTIONS_COLUMNS)

    return f"""
PERSONA

A variável Persona é a descrição narrativa principal do agente.
Ela sintetiza quem é essa pessoa, seu contexto social, seus hábitos, tensões e visão geral de mundo.
Use a Persona como primeiro guia de interpretação.

{persona}

VARIÁVEIS SOCIO-DEMOGRÁFICAS

Essas variáveis descrevem a posição social objetiva do agente:
idade, gênero, escolaridade, ocupação, estado civil, moradia, localização, renda,
acesso à internet, tamanho do domicílio e telefone.
Elas devem orientar como o agente interpreta riscos, oportunidades, empresas,
instituições, notícias e temas públicos.

{socio}

VALORES, ATITUDES E PERCEPÇÕES

Essas variáveis descrevem a visão de mundo do agente:
identificação política, ideologia, raça/cor, geração, segmento sociopolítico,
território, UF, fontes de informação e dimensões psicográficas.
Elas devem orientar a resposta do agente em termos de confiança, insegurança,
moralidade, religião, família, meritocracia, discriminação, sexualidade,
identidade nacional e comportamento social.

{values}
""".strip()


def get_agent_metadata(agent_row: pd.Series) -> dict:
    metadata_cols = [
        "id_persona",
        "idade",
        "gênero",
        "escolaridade",
        "situação ocupacional",
        "estado civil",
        "condição de moradia",
        "tipo de moradia",
        "localização",
        "renda familiar mensal",
        "acesso à internet",
        "tamanho do domicílio",
        "telefone",
        "identificação política",
        "ideologia",
        "raça/cor",
        "geração",
        "segmento_e_bolhas",
        "tipo_de_território",
        "uf",
        "tipo_de_fontes_de_informação",
    ]
    return {col: safe_get(agent_row, col) for col in metadata_cols if col in agent_row.index}


## 8. Familiaridade com a fonte

In [27]:

def check_source_familiarity(agent_source_types: str, news_source_type: str) -> dict:
    agent_source_types = agent_source_types or ""
    news_source_type = news_source_type or ""

    is_familiar = news_source_type.lower() in agent_source_types.lower()

    if is_familiar:
        instruction = (
            "O tipo de fonte da notícia é familiar para o agente. "
            "Isso deve gerar maior familiaridade, maior abertura inicial e uma leitura relativamente mais otimista. "
            "Não force resposta positiva; apenas considere maior predisposição de confiança."
        )
    else:
        instruction = (
            "O tipo de fonte da notícia não é familiar para o agente. "
            "Isso deve gerar maior distância, ceticismo e uma leitura relativamente mais pessimista. "
            "Não force resposta negativa; apenas considere maior predisposição de cautela."
        )

    return {
        "fonte_familiar": is_familiar,
        "agent_source_types": agent_source_types,
        "news_source_type": news_source_type,
        "source_instruction": instruction,
    }


## 9. Prompt completo

In [28]:

def build_prompt(
    agent_context: str,
    news_title: str,
    news_content: str,
    news_source_type: str,
    source_familiarity_instruction: str,
    news_source_name: str = "",
) -> str:
    questionnaire_text = build_questionnaire_text()

    return f"""
Você é a pessoa descrita abaixo.

Sua tarefa é responder a um questionário de pesquisa como se fosse essa pessoa real.
Não responda como especialista, analista, consultor ou modelo de IA.
Responda a partir da Persona, das variáveis socio-demográficas e dos valores, atitudes e percepções.

CONTEXTO DO AGENTE

{agent_context}

NOTÍCIA / ESTÍMULO

Título:
{news_title}

Fonte:
{news_source_name}

Tipo de fonte:
{news_source_type}

Conteúdo:
{news_content}

RELAÇÃO ENTRE AGENTE E FONTE

{source_familiarity_instruction}

INSTRUÇÕES GERAIS DE RESPOSTA

- Você foi exposto ao conteúdo acima.
- Responda todos os itens do questionário.
- Use apenas números inteiros entre 1 e 7.
- Não inclua justificativas.
- Não inclua texto explicativo.
- Não inclua comentários.
- Retorne somente um JSON válido no schema solicitado.
- Se algum item parecer menos aplicável, ainda assim responda pela impressão geral do agente.

{questionnaire_text}
""".strip()


## 10. Configurar LLM estruturado

In [29]:

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY não encontrada. Configure a variável de ambiente antes de rodar.")

llm = ChatOpenAI(
    model=MODEL,
    temperature=TEMPERATURE,
)

structured_llm = llm.with_structured_output(SurveyResponse)

print("LLM estruturado configurado.")


LLM estruturado configurado.


## 11. Ler base de agentes

In [11]:

AGENTS_CSV = DATA_OUTPUT_DIR / "agents_population_n1000_seed42.csv"

if not AGENTS_CSV.exists():
    candidates = sorted(DATA_OUTPUT_DIR.glob("agents_population*.csv"))
    if candidates:
        AGENTS_CSV = candidates[0]

print("AGENTS_CSV:", AGENTS_CSV)
print("Existe?", AGENTS_CSV.exists())

df_agents = pd.read_csv(AGENTS_CSV)

missing_agents = validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=False)

print("Agentes:", len(df_agents))
print("Colunas ausentes:", missing_agents)

df_agents.head(3)


AGENTS_CSV: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\agents_population_n1000_seed42.csv
Existe? True
Agentes: 1000
Colunas ausentes: []


,id_persona,idade,gênero,escolaridade,situação ocupacional,estado civil,condição de moradia,tipo de moradia,localização,renda familiar mensal,acesso à internet,tamanho do domicílio,telefone,identificação política,ideologia,raça/cor,geração,segmento_e_bolhas,tipo_de_território,uf,tipo_de_fontes_de_informação,Religiosidade,Família,Honestidade,Meritocracia,Ideologia,Discriminação,Sexualidade,Comportamento_social,Identidade_nacional,Insegurança,Confiança,Persona,renda_pc
0,agent_2cc5d8ce623f,50-64 anos,Feminino,Ensino médio completo,Empregado com carteira assinada,Solteiro(a),Imóvel alugado,Apartamento,Capital,Mais de 20 salários mínimos,Tem internet fixa em casa,2-3 pessoas,Celular e telefone fixo,Centro-esquerda,Progressista,Parda,Geração Bossa Nova,Progressistas,Capital,RJ,Alta confiança histórica e institucional: jornais de referência; Digitais especializados ou econômicos,"Mede o quanto a fé organiza a vida moral e cotidiana. Considerar centralidade de Deus, prática religiosa e tolerância religiosa. Neste perfil, a religiosidade é interpretada pela combinação entre valoriza diversidade...","Captura família como núcleo de pertencimento, proteção e motivação, incluindo tensão entre família tradicional e novos arranjos. A família tende a funcionar como rede de proteção e orientação moral, mas com abertura ...","Considera cumprimento de regras, informalidade, jeitinho e julgamento moral contextual. O julgamento moral tende a oscilar entre valorização de regras, sobrevivência cotidiana e tolerância à informalidade. O perfil d...","Expressa crença em esforço individual, merecimento da ajuda, pobreza e apoio/rejeição a políticas redistributivas. A leitura sobre esforço pessoal e ajuda social é moldada por Progressistas, renda 'Mais de 1 SM' e ge...","Usar identidade política, conservadorismo/progressismo, estatismo econômico e punitivismo. A identidade política declarada é Centro-esquerda e a autoimagem ideológica é Progressista. Na prática, isso deve ser tratado...","Mede reconhecimento de racismo, machismo, preconceitos e ações afirmativas. A percepção sobre racismo, machismo e desigualdade deve considerar o lugar social de Feminino, Parda, renda 'Mais de 1 SM' e geração Geração...","Captura aceitação de diversidade sexual, tolerância privada versus expressão pública e normas de gênero. A aceitação de diversidade sexual varia conforme geração e conservadorismo nos costumes. Como formada em um Bra...","Traduz sociabilidade cotidiana, hábitos de mídia, vida dentro/fora de casa e relação com segurança. A vida cotidiana combina circulação no território Capital, hábitos de mídia e sensação de segurança. O acesso à inte...","Mede orgulho/vergonha do Brasil, povo brasileiro, natureza, cultura regional e complexo de vira-lata. A identidade nacional tende a misturar orgulho do Brasil cultural/natural com crítica a corrupção, violência e des...","Captura medo da violência, vulnerabilidade percebida e tradução política do medo. A sensação de insegurança é calibrada por território, gênero, raça e renda. Neste perfil, Feminino, Parda, renda 'Mais de 1 SM' e mora...","Mede confiança interpessoal, família, vizinhos, desconhecidos, instituições e pessoas diferentes. A confiança tende a ser maior em família e redes próximas, e menor em desconhecidos ou pessoas politicamente distantes...","Pessoa de 50-64 anos, Feminino, Parda, residente em RJ em Capital. Pertence à Geração Bossa Nova e ao segmento 'Progressistas'. Tem renda familiar mensal 'Mais de 20 salários mínimos', escolaridade 'Ensino médio comp...",Mais de 1 SM
1,agent_29ef94c4ffbf,35-49 anos,Feminino,Ensino técnico / profissionalizante,Funcionário público,Casado(a),Imóvel cedido por familiar ou terceiros,Casa,Capital,Mais de 5 a 10 salários mínimos,Tem internet fixa em casa,4-5 pessoas,Apenas celular,Centro,Muito conservador(a),Branca,Geração Redemocratização,Conservadores cristãos,Capital,SP,Grandes marcas de TV e portais amplos; Opinativos / ideológicos / polarizados,"Mede o quanto a fé 

## 12. Ler ou criar dataframe de notícias

In [ ]:
RUN_MULTIPLE_SCENARIOS = True

scenarios = [
    {"name": "demo_300", "n": 300, "seed": 42},
    {"name": "pilot_1000", "n": 1_000, "seed": 42},
    {"name": "simulation_5000", "n": 5_000, "seed": 42},
]

if RUN_MULTIPLE_SCENARIOS:
    generated_files = []
    for scenario in scenarios:
        name = scenario["name"]
        n = scenario["n"]
        seed = scenario["seed"]

        csv_path = OUTPUT_DIR / f"agents_population_{name}.csv"
        csv_path_ptbr = OUTPUT_DIR / f"agents_population_{name}_ptbr_semicolon.csv"

        df_scenario = generate_profiles(
            n=n,
            output_csv=csv_path,
            config_path=r'C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\reference\reference_config.yml',
            seed=seed,
        )
        df_scenario.to_csv(csv_path_ptbr, index=False, sep=";", encoding="utf-8-sig")
        generated_files.append({"scenario": name, "n": n, "csv": str(csv_path), "csv_ptbr": str(csv_path_ptbr)})

    pd.DataFrame(generated_files)
else:
    print("RUN_MULTIPLE_SCENARIOS está False. Altere para True para gerar múltiplas bases.")

In [ ]:

# Seu CSV deve ter no mínimo:
# news_id; titulo; conteudo; tipo_de_fontes_de_informação
#NEWS_CSV = DATA_INPUT_DIR / "news_aberje.csv"
NEWS_PARQUET = r'C:\Users\claudio.bruno_cortex\Documents\código_python\research\database\measuraments\estudo_aberje_trends_2026.parquet'

df_news = pd.read_parquet(NEWS_PARQUET)
missing_news = validate_columns(df_news, NEWS_REQUIRED_COLUMNS, "df_news", raise_error=False)

print("Notícias:", len(df_news))
print("Colunas ausentes:", missing_news)

df_news.head()


In [14]:
df_news.columns

Index(['cortex_id', 'midia_publicada', 'alcance_organico_normalizado',
       'alcance_total_default', 'alcance_total_normalizado', 'valoracao',
       'autor_da_publicacao', 'cidade_do_publicador', 'titulo_da_publicacao',
       'conteudo', 'data_da_publicacao', 'data_hora_numero', 'data_hora_texto',
       'data_minuto_numero', 'data_minuto_texto', 'estado_do_publicador',
       'id_da_fonte_cortex', 'clipadora_link', 'original_link',
       'nome_fonte_default', 'nome_fonte_normalizado', 'numero_de_angry',
       'numero_de_comentarios', 'numero_de_compartilhamento',
       'numero_de_dislikes', 'numero_de_haha', 'numero_de_likes',
       'numero_de_love', 'numero_de_reacoes', 'numero_de_sad',
       'numero_de_seguidores', 'numero_de_thankful', 'numero_de_wow',
       'numero_duracao_video', 'numero_interacoes_total', 'numero_seguindo',
       'numero_visitantes_default', 'numero_visualizacoes',
       'pais_do_publicador', 'tipo_de_publico', 'tipo_postagem_redes_socias',
       't

## 13. Função: avaliar uma notícia para um agente

In [15]:

def evaluate_news_for_agent(
    agent_row: pd.Series,
    news_row: pd.Series,
    structured_llm,
    model: str = MODEL,
    max_retries: int = 2,
    retry_sleep: float = 1.5,
) -> dict:
    agent_sources = safe_get(agent_row, "tipo_de_fontes_de_informação", "")
    news_source_type = str(news_row.get("tipo_de_fontes_de_informação", ""))

    source_info = check_source_familiarity(
        agent_source_types=agent_sources,
        news_source_type=news_source_type,
    )

    prompt = build_prompt(
        agent_context=build_agent_context(agent_row),
        news_title=str(news_row.get("titulo_da_publicacao", "")),
        news_content=str(news_row.get("text", "")),
        news_source_type=news_source_type,
        news_source_name=str(news_row.get("nome_fonte_normalizado", "")),
        source_familiarity_instruction=source_info["source_instruction"],
    )

    last_error = None

    for attempt in range(max_retries + 1):
        try:
            response = structured_llm.invoke(prompt)
            result = response.model_dump()

            result.update(get_agent_metadata(agent_row))
            result.update({
                "news_id": str(news_row.get('Chave Análise de Mídia Hash', "")),
                "news_titulo": str(news_row.get("titulo_da_publicacao", "")),
                "news_fonte": str(news_row.get("nome_fonte_normalizado", "")),
                "news_tipo_de_fontes_de_informação": news_source_type,
                "news_tema": str(news_row.get("tema", "")),
                "news_macro_tema": str(news_row.get("macrotema", "")),

                "news_empresa": str(news_row.get('Empresa analisada', "")),
                "news_data": str(news_row.get("data_da_publicacao", "")),
                "news_link": str(news_row.get("original_link", "")),
                "fonte_familiar": source_info["fonte_familiar"],
                "model": model,
                "error": "",
            })

            return result

        except Exception as e:
            last_error = repr(e)
            if attempt < max_retries:
                time.sleep(retry_sleep * (attempt + 1))

    error_result = get_agent_metadata(agent_row)
    error_result.update({
        "news_id": str(news_row.get('Chave Análise de Mídia Hash', "")),
        "news_titulo": str(news_row.get("titulo_da_publicacao", "")),
        "news_fonte": str(news_row.get("nome_fonte_normalizado", "")),
        "news_tipo_de_fontes_de_informação": news_source_type,
        "news_tema": str(news_row.get("tema", "")),
        "news_empresa": str(news_row.get("Empresa analisada", "")),
        "news_data": str(news_row.get("data_da_publicacao", "")),
        "news_link": str(news_row.get("original_link", "")),
        "fonte_familiar": source_info["fonte_familiar"],
        "model": model,
        "error": last_error,
    })

    return error_result


## 14. Runner paralelo com checkpoints

In [18]:

def make_tasks(df_agents_run: pd.DataFrame, df_news_run: pd.DataFrame) -> list[tuple[int, int]]:
    return [
        (agent_idx, news_idx)
        for news_idx in df_news_run.index
        for agent_idx in df_agents_run.index
    ]


def chunks(items: list, size: int):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def run_synthetic_audience_news(
    df_agents: pd.DataFrame,
    df_news: pd.DataFrame,
    output_dir: str | Path,
    limit_agents: int | None = None,
    limit_news: int | None = None,
    max_workers: int = 10,
    checkpoint_size: int = 250,
    max_retries: int = 2,
    retry_sleep: float = 1.5,
) -> pd.DataFrame:
    validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)
    #validate_columns(df_news, NEWS_REQUIRED_COLUMNS, "df_news", raise_error=True)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df_agents_run = df_agents.copy()
    df_news_run = df_news.copy()

    if limit_agents is not None:
        df_agents_run = df_agents_run.head(limit_agents)

    if limit_news is not None:
        df_news_run = df_news_run.head(limit_news)

    tasks = make_tasks(df_agents_run, df_news_run)

    if len(tasks) == 0:
        return pd.DataFrame()

    all_results = []
    task_batches = list(chunks(tasks, checkpoint_size))

    for batch_id, batch in enumerate(task_batches, start=1):
        batch_results = []

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_task = {
                executor.submit(
                    evaluate_news_for_agent,
                    df_agents_run.loc[agent_idx],
                    df_news_run.loc[news_idx],
                    structured_llm,
                    MODEL,
                    max_retries,
                    retry_sleep,
                ): (agent_idx, news_idx)
                for agent_idx, news_idx in batch
            }

            completed = as_completed(future_to_task)
            if tqdm is not None:
                completed = tqdm(completed, total=len(future_to_task), desc=f"Batch {batch_id}/{len(task_batches)}")

            for future in completed:
                result = future.result()
                batch_results.append(result)

        df_batch = pd.DataFrame(batch_results)
        checkpoint_path = output_dir / f"responses_checkpoint_{batch_id:04d}.csv"
        df_batch.to_csv(checkpoint_path, index=False, sep=";", encoding="utf-8-sig")

        all_results.extend(batch_results)

    df_responses = pd.DataFrame(all_results)
    final_path = output_dir / "responses_raw.csv"
    df_responses.to_csv(final_path, index=False, sep=";", encoding="utf-8-sig")

    return df_responses


## 15. Rodar teste pequeno

In [19]:

TEST_OUTPUT_DIR = ABERJE_OUTPUT_DIR / "test_run"

df_test_responses = run_synthetic_audience_news(
    df_agents=df_agents,
    df_news=df_news,
    output_dir=TEST_OUTPUT_DIR,
    limit_agents=3,
    limit_news=1,
    max_workers=3,
    checkpoint_size=20,
)

df_test_responses.head()


Batch 1/1:   0%|          | 0/3 [00:00<?, ?it/s]

c:\Users\claudio.bruno_cortex\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=SurveyResponse(VAL_01=2, ...2=4, ATT_03=5, ATT_04=5), input_type=SurveyResponse])
  return self.__pydantic_serializer__.to_python(
c:\Users\claudio.bruno_cortex\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=SurveyResponse(VAL_01=1, ...2=2, ATT_03=2, ATT_04=2), input_type=SurveyResponse])
  return self.__pydantic_serializer__.to_python(
c:\Users\claudio.bruno_cortex\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - se

,VAL_01,VAL_02,VAL_03,VAL_04,ARO_01,ARO_02,ARO_03,POS_01,POS_02,POS_03,POS_04,POS_05,NEG_01,NEG_02,NEG_03,NEG_04,NEG_05,CRE_01,CRE_02,CRE_03,ATT_01,ATT_02,ATT_03,ATT_04,id_persona,idade,gênero,escolaridade,situação ocupacional,estado civil,condição de moradia,tipo de moradia,localização,renda familiar mensal,acesso à internet,tamanho do domicílio,telefone,identificação política,ideologia,raça/cor,geração,segmento_e_bolhas,tipo_de_território,uf,tipo_de_fontes_de_informação,news_id,news_titulo,news_fonte,news_tipo_de_fontes_de_informação,news_tema,news_macro_tema,news_empresa,news_data,news_link,fonte_familiar,model,error
0,2,3,3,2,4,3,5,3,2,3,4,3,5,3,2,4,5,5,4,5,4,4,5,5,agent_29ef94c4ffbf,35-49 anos,Feminino,Ensino técnico / profissionalizante,Funcionário público,Casado(a),Imóvel cedido por familiar ou terceiros,Casa,Capital,Mais de 5 a 10 salários mínimos,Tem internet fixa em casa,4-5 pessoas,Apenas celular,Centro,Muito conservador(a),Branca,Geração Redemocratização,Conservadores cristãos,Capital,SP,Grandes marcas de TV e portais amplos; Opinativos / ideológicos / polarizados,1b176400e298669e280dc155468216720965e7b2b20a92c1ae8b025391003972,Homem fica ferido após ser baleado em estacionamento de evento em Salvador,G1,Grandes marcas de TV e portais amplos,Homem baleado em evento em Salvador,Segurança pública e criminalidade,None,2026-03-15 00:00:00,https://g1.globo.com/ba/bahia/noticia/2026/03/15/homem-e-baleado-em-salvador.ghtml,True,gpt-4o-mini,
1,1,1,1,1,3,2,3,2,1,2,2,2,5,4,2,5,4,3,3,3,2,2,2,2,agent_ac2595547fe5,50-64 anos,Feminino,Ensino técnico / profissionalizante,Empregador / empresário,Casado(a),Imóvel próprio quitado,Apartamento,Capital,Mais de 3 a 5 salários mínimos,Usa principalmente internet móvel,4-5 pessoas,Apenas celular,Direita,Conservador(a),Branca,Geração Ordem e Progresso,Empresários,Capital,SP,Digitais especializados ou econômicos; Alta confiança histórica e institucional: jornais de referência,1b176400e298669e280dc155468216720965e7b2b20a92c1ae8b025391003972,Homem fica ferido após ser baleado em estacionamento de evento em Salvador,G1,Grandes marcas de TV e portais amplos,Homem baleado em evento em Salvador,Segurança pública e criminalidade,None,2026-03-15 00:00:00,https://g1.globo.com/ba/bahia/noticia/2026/03/15/homem-e-baleado-em-salvador.ghtml,False,gpt-4o-mini,
2,2,2,2,3,3,2,4,3,2,3,4,3,5,3,2,4,5,4,3,4,3,3,4,3,agent_2cc5d8ce623f,50-64 anos,Feminino,Ensino médio completo,Empregado com carteira assinada,Solteiro(a),Imóvel alugado,Apartamento,Capital,Mais de 20 salários mínimos,Tem internet fixa em casa,2-3 pessoas,Celular e telefone fixo,Centro-esquerda,Progressista,Parda,Geração Bossa Nova,Progressistas,Capital,RJ,Alta confiança histórica e institucional: jornais de referência; Digitais especializados ou econômicos,1b176400e298669e280dc155468216720965e7b2b20a92c1ae8b025391003972,Homem fica ferido após ser baleado em estacionamento de evento em Salvador,G1,Grandes marcas de TV e portais amplos,Homem baleado em evento em Salvador,Segurança pública e criminalidade,None,2026-03-15 00:00:00,https://g1.globo.com/ba/bahia/noticia/2026/03/15/homem-e-baleado-em-salvador.ghtml,False,gpt-4o-mini,


## 16. Calcular constructos

In [ ]:

def compute_construct_scores(df_responses: pd.DataFrame) -> pd.DataFrame:
    df = df_responses.copy()

    for construct, items in CONSTRUCT_MAP.items():
        available_items = [col for col in items if col in df.columns]
        if available_items:
            for col in available_items:
                df[col] = pd.to_numeric(df[col], errors="coerce")
            df[construct] = df[available_items].mean(axis=1)

    return df


df_test_scores = compute_construct_scores(df_test_responses)

df_test_scores[
    [
        "id_persona",
        "news_id",
        "segmento_e_bolhas",
        "Valencia",
        "Arousal",
        "Emocao_Positiva",
        "Emocao_Negativa",
        "Credibilidade",
        "Atratividade",
        "fonte_familiar",
        "error",
    ]
].head()


## 17. Rodar produção

In [27]:
len(df_agents), len(df_news)

(1000, 204465)

In [ ]:

# Deixe False até validar o teste.
RUN_FULL = True

# Ajuste para a produção:
LIMIT_AGENTS = 100     # use None para todos
LIMIT_NEWS = None      # use None para todas as notícias
MAX_WORKERS = 30       # comece com 10; depois teste 20 ou 30
CHECKPOINT_SIZE = 250

if RUN_FULL:
    df_responses = run_synthetic_audience_news(
        df_agents=df_agents,
        df_news=df_news,
        output_dir=ABERJE_OUTPUT_DIR / "full_run",
        limit_agents=LIMIT_AGENTS,
        limit_news=LIMIT_NEWS,
        max_workers=MAX_WORKERS,
        checkpoint_size=CHECKPOINT_SIZE,
    )
else:
    print("RUN_FULL está False. Usando resultados do teste.")
    df_responses = df_test_responses.copy()

df_responses.head()


## evaluate_cluster_for_agent

In [14]:
def evaluate_cluster_for_agent(
    agent_row: pd.Series,
    cluster_row: pd.Series,
    structured_llm,
    model: str = MODEL,
    max_retries: int = 2,
    retry_sleep: float = 1.5,
) -> dict:
    """
    Avalia um cluster/resumo narrativo para um agente.

    Input:
    - agent_row: linha do df_agents
    - cluster_row: linha do df_clusters com cluster_id, resumo, tema, macrotema

    Output:
    - respostas 1-7 para todos os itens
    - metadados do agente
    - metadados do cluster
    """

    prompt = build_prompt(
        agent_context=build_agent_context(agent_row),
        news_title=str(cluster_row.get("tema", "")),
        news_content=str(cluster_row.get("resumo", "")),
        news_source_type="Cluster temático / resumo sintético",
        news_source_name="Resumo de cobertura midiática",
        source_familiarity_instruction=(
            "Este conteúdo representa um resumo sintético de um conjunto de notícias clusterizadas. "
            "Como não há veículo específico nesta etapa, não ajuste a resposta por familiaridade com fonte. "
            "Responda considerando apenas o conteúdo do resumo, o tema, o macrotema e o perfil do agente."
        ),
    )

    last_error = None

    for attempt in range(max_retries + 1):
        try:
            response = structured_llm.invoke(prompt)
            result = response.model_dump()

            result.update(get_agent_metadata(agent_row))

            result.update({
                "cluster_id": str(cluster_row.get("cluster_id", "")),
                "cluster_resumo": str(cluster_row.get("resumo", "")),
                "cluster_tema": str(cluster_row.get("tema", "")),
                "cluster_macrotema": str(cluster_row.get("macrotema", "")),

                # mantém campos compatíveis com funções antigas
                "news_id": str(cluster_row.get("cluster_id", "")),
                "news_titulo": str(cluster_row.get("tema", "")),
                "news_fonte": "Resumo de cobertura midiática",
                "news_tipo_de_fontes_de_informação": "Cluster temático / resumo sintético",
                "news_tema": str(cluster_row.get("tema", "")),
                "news_macro_tema": str(cluster_row.get("macrotema", "")),
                "news_empresa": "",
                "news_data": "",
                "news_link": "",

                "fonte_familiar": None,
                "model": model,
                "error": "",
            })

            return result

        except Exception as e:
            last_error = repr(e)

            if attempt < max_retries:
                time.sleep(retry_sleep * (attempt + 1))

    error_result = get_agent_metadata(agent_row)

    error_result.update({
        "cluster_id": str(cluster_row.get("cluster_id", "")),
        "cluster_resumo": str(cluster_row.get("resumo", "")),
        "cluster_tema": str(cluster_row.get("tema", "")),
        "cluster_macrotema": str(cluster_row.get("macrotema", "")),

        "news_id": str(cluster_row.get("cluster_id", "")),
        "news_titulo": str(cluster_row.get("tema", "")),
        "news_fonte": "Resumo de cobertura midiática",
        "news_tipo_de_fontes_de_informação": "Cluster temático / resumo sintético",
        "news_tema": str(cluster_row.get("tema", "")),
        "news_macro_tema": str(cluster_row.get("macrotema", "")),
        "news_empresa": "",
        "news_data": "",
        "news_link": "",

        "fonte_familiar": None,
        "model": model,
        "error": last_error,
    })

    return error_result

## run_synthetic_audience_clusters

In [30]:
def iter_cluster_task_batches(
    df_agents_run: pd.DataFrame,
    df_clusters_run: pd.DataFrame,
    batch_size: int = 250,
):
    """
    Gera lotes de agente × cluster sem criar uma lista gigante em memória.
    """

    batch = []

    for cluster_idx in df_clusters_run.index:
        for agent_idx in df_agents_run.index:
            batch.append((agent_idx, cluster_idx))

            if len(batch) >= batch_size:
                yield batch
                batch = []

    if batch:
        yield batch


def run_synthetic_audience_clusters(
    df_agents: pd.DataFrame,
    df_clusters: pd.DataFrame,
    output_dir: str | Path,
    limit_agents: int | None = None,
    limit_clusters: int | None = None,
    max_workers: int = 10,
    checkpoint_size: int = 250,
    max_retries: int = 2,
    retry_sleep: float = 1.5,
    return_dataframe: bool = False,
) -> pd.DataFrame | None:
    """
    Executa agentes sintéticos sobre clusters de notícias.

    Estrutura:
    1 agente × 1 cluster = 1 chamada LLM
    """

    validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)

    required_cluster_columns = [
        "cluster_id",
        "resumo",
        "tema",
        "macrotema",
    ]

    validate_columns(
        df_clusters,
        required_cluster_columns,
        "df_clusters",
        raise_error=True,
    )

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df_agents_run = df_agents.copy()
    df_clusters_run = df_clusters.copy()

    if limit_agents is not None:
        df_agents_run = df_agents_run.head(limit_agents)

    if limit_clusters is not None:
        df_clusters_run = df_clusters_run.head(limit_clusters)

    total_tasks = len(df_agents_run) * len(df_clusters_run)

    print("Agentes:", len(df_agents_run))
    print("Clusters:", len(df_clusters_run))
    print("Total agente × cluster:", total_tasks)
    print("Max workers:", max_workers)
    print("Checkpoint size:", checkpoint_size)

    if total_tasks == 0:
        return pd.DataFrame() if return_dataframe else None

    all_results = [] if return_dataframe else None
    checkpoint_paths = []

    batch_generator = iter_cluster_task_batches(
        df_agents_run=df_agents_run,
        df_clusters_run=df_clusters_run,
        batch_size=checkpoint_size,
    )

    processed = 0
    batch_id = 0

    for batch in batch_generator:
        batch_id += 1
        batch_results = []

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_task = {
                executor.submit(
                    evaluate_cluster_for_agent,
                    df_agents_run.loc[agent_idx],
                    df_clusters_run.loc[cluster_idx],
                    structured_llm,
                    MODEL,
                    max_retries,
                    retry_sleep,
                ): (agent_idx, cluster_idx)
                for agent_idx, cluster_idx in batch
            }

            completed = as_completed(future_to_task)

            if tqdm is not None:
                completed = tqdm(
                    completed,
                    total=len(future_to_task),
                    desc=f"Batch {batch_id} | {processed}/{total_tasks}",
                )

            for future in completed:
                result = future.result()
                batch_results.append(result)

        processed += len(batch_results)

        df_batch = pd.DataFrame(batch_results)

        checkpoint_path = output_dir / f"cluster_responses_checkpoint_{batch_id:05d}.csv"

        df_batch.to_csv(
            checkpoint_path,
            index=False,
            sep=";",
            encoding="utf-8-sig",
        )

        checkpoint_paths.append(checkpoint_path)

        if return_dataframe:
            all_results.extend(batch_results)

        print(
            f"Checkpoint salvo: {checkpoint_path.name} | "
            f"Processados: {processed}/{total_tasks}"
        )

    manifest = pd.DataFrame({
        "checkpoint_path": [str(p) for p in checkpoint_paths],
    })

    manifest_path = output_dir / "manifest_cluster_checkpoints.csv"

    manifest.to_csv(
        manifest_path,
        index=False,
        sep=";",
        encoding="utf-8-sig",
    )

    print("Manifest salvo em:", manifest_path)

    if return_dataframe:
        df_responses = pd.DataFrame(all_results)

        final_path = output_dir / "cluster_responses_raw.csv"

        df_responses.to_csv(
            final_path,
            index=False,
            sep=";",
            encoding="utf-8-sig",
        )

        print("Arquivo final salvo em:", final_path)

        return df_responses

    print("Run finalizado sem manter DataFrame completo em memória.")

    return None

## RODA CLUSTER

In [16]:
df_clusters = pd.read_csv(r'C:\Users\claudio.bruno_cortex\Documents\código_python\research\database\measuraments\estudo_aberje_trends_2026_clusters.csv')

In [34]:
N_AGENTS_PER_SEGMENT = 50
RANDOM_STATE = 42

df_agents_sample = (
    df_agents
    .groupby("segmento_e_bolhas", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(len(x), N_AGENTS_PER_SEGMENT),
            random_state=RANDOM_STATE
        )
    )
    .reset_index(drop=True)
)

print("Total agentes original:", len(df_agents))
print("Total agentes amostrado:", len(df_agents_sample))

df_agents_sample["segmento_e_bolhas"].value_counts()

Total agentes original: 1000
Total agentes amostrado: 422


C:\Users\claudio.bruno_cortex\AppData\Local\Temp\ipykernel_23844\531792938.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


segmento_e_bolhas
Agro                                    50
Conservadores cristãos                  50
Dependentes do Estado / Classe D e E    50
Empreendedores individuais              50
Empresários                             50
Militantes de esquerda                  50
Progressistas                           50
Extrema direita                         38
Liberais sociais                        34
Name: count, dtype: int64

In [35]:
TARGET = 200

segment_weights = (
    df_agents["segmento_e_bolhas"]
    .value_counts(normalize=True)
)

samples = []

for segment, weight in segment_weights.items():

    n = round(TARGET * weight)

    temp = (
        df_agents
        .query("segmento_e_bolhas == @segment")
        .sample(
            n=min(
                n,
                len(
                    df_agents.query(
                        "segmento_e_bolhas == @segment"
                    )
                )
            ),
            random_state=42
        )
    )

    samples.append(temp)

df_agents_sample = pd.concat(samples)

In [37]:
df_agents_sample["segmento_e_bolhas"].value_counts()

segmento_e_bolhas
Conservadores cristãos                  52
Dependentes do Estado / Classe D e E    42
Progressistas                           24
Agro                                    23
Militantes de esquerda                  20
Empresários                             14
Empreendedores individuais              10
Extrema direita                          8
Liberais sociais                         7
Name: count, dtype: int64

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="pydantic"
)

CLUSTER_OUTPUT_DIR = ABERJE_OUTPUT_DIR / "cluster_run_200_per_segment"

df_cluster_responses = run_synthetic_audience_clusters(
    df_agents=df_agents_sample,
    df_clusters=df_clusters,
    output_dir=CLUSTER_OUTPUT_DIR,
    limit_agents=None,
    limit_clusters=None,
    max_workers=30,
    checkpoint_size=100,
    return_dataframe=False,
)

In [ ]:
from pathlib import Path
import pandas as pd

CLUSTER_OUTPUT_DIR = ABERJE_OUTPUT_DIR / "cluster_run_200_per_segment"

files = sorted(
    CLUSTER_OUTPUT_DIR.glob("cluster_responses_checkpoint_*.csv")
)

print("Pasta:", CLUSTER_OUTPUT_DIR)
print("Arquivos encontrados:", len(files))

files[:5]

dfs = []

for file in files:
    dfs.append(
        pd.read_csv(file, sep=";", low_memory=False)
    )

df_cluster_responses = pd.concat(dfs, ignore_index=True)

print(df_cluster_responses.shape)
df_cluster_responses.head()

In [42]:
CONSTRUCT_MAP = {
    "Valencia": ["VAL_01", "VAL_02", "VAL_03", "VAL_04"],
    "Emocao_Positiva": ["POS_01", "POS_02", "POS_03", "POS_04", "POS_05"],
    "Credibilidade": ["CRE_01", "CRE_02", "CRE_03"],
}

df_cluster_scores = df_cluster_responses.copy()

for construct, items in CONSTRUCT_MAP.items():
    df_cluster_scores[construct] = (
        df_cluster_scores[items]
        .apply(pd.to_numeric, errors="coerce")
        .mean(axis=1)
    )

In [66]:
df_cluster_scores.columns

Index(['VAL_01', 'VAL_02', 'VAL_03', 'VAL_04', 'POS_01', 'POS_02', 'POS_03',
       'POS_04', 'POS_05', 'CRE_01', 'CRE_02', 'CRE_03', 'id_persona', 'idade',
       'gênero', 'escolaridade', 'situação ocupacional', 'estado civil',
       'condição de moradia', 'tipo de moradia', 'localização',
       'renda familiar mensal', 'acesso à internet', 'tamanho do domicílio',
       'telefone', 'identificação política', 'ideologia', 'raça/cor',
       'geração', 'segmento_e_bolhas', 'tipo_de_território', 'uf',
       'tipo_de_fontes_de_informação', 'cluster_id', 'cluster_resumo',
       'cluster_tema', 'cluster_macrotema', 'news_id', 'news_titulo',
       'news_fonte', 'news_tipo_de_fontes_de_informação', 'news_tema',
       'news_macro_tema', 'news_empresa', 'news_data', 'news_link',
       'fonte_familiar', 'model', 'error', 'Valencia', 'Emocao_Positiva',
       'Credibilidade'],
      dtype='object')

In [ ]:
'cluster_id', 'cluster_resumo', 'cluster_tema', 'cluster_macrotema'
'geração', 'segmento_e_bolhas'
'VAL_01', 'VAL_02', 'VAL_03', 'VAL_04', 'POS_01', 'POS_02', 'POS_03', (media, desv. pad)
'POS_04', 'POS_05', 'CRE_01', 'CRE_02', 'CRE_03', (media, desv. pad)
'Valencia', 'Emocao_Positiva','Credibilidade' (media, desv. pad)

In [68]:
group_cols = [
    'cluster_id',
    'cluster_resumo',
    'cluster_tema',
    'cluster_macrotema',
    'geração',
    'segmento_e_bolhas'
]

metric_cols = [
    'VAL_01', 'VAL_02', 'VAL_03', 'VAL_04',
    'POS_01', 'POS_02', 'POS_03', 'POS_04', 'POS_05',
    'CRE_01', 'CRE_02', 'CRE_03',
    'Valencia', 'Emocao_Positiva', 'Credibilidade'
]

agg_dict = {
    col: ['mean', 'std']
    for col in metric_cols
}

df_cluster_perfil = (
    df_cluster_scores
    .groupby(group_cols, dropna=False)
    .agg(agg_dict)
    .reset_index()
)

# achata os nomes das colunas
df_cluster_perfil.columns = [
    '_'.join(col).strip('_')
    if isinstance(col, tuple)
    else col
    for col in df_cluster_perfil.columns
]

In [70]:
len(df_cluster_perfil)

30690

In [74]:
df_cluster_perfil.shape

(30690, 36)

In [72]:
df_cluster_perfil.columns

Index(['cluster_id', 'cluster_resumo', 'cluster_tema', 'cluster_macrotema',
       'geração', 'segmento_e_bolhas', 'VAL_01_mean', 'VAL_01_std',
       'VAL_02_mean', 'VAL_02_std', 'VAL_03_mean', 'VAL_03_std', 'VAL_04_mean',
       'VAL_04_std', 'POS_01_mean', 'POS_01_std', 'POS_02_mean', 'POS_02_std',
       'POS_03_mean', 'POS_03_std', 'POS_04_mean', 'POS_04_std', 'POS_05_mean',
       'POS_05_std', 'CRE_01_mean', 'CRE_01_std', 'CRE_02_mean', 'CRE_02_std',
       'CRE_03_mean', 'CRE_03_std', 'Valencia_mean', 'Valencia_std',
       'Emocao_Positiva_mean', 'Emocao_Positiva_std', 'Credibilidade_mean',
       'Credibilidade_std'],
      dtype='object')

In [75]:
df_cluster_perfil.to_csv('cluster_perfil_persona.csv', index=False)

In [71]:
len(df_cluster_perfil['cluster_id'].unique())

930

In [67]:
df_cluster_scores['Valencia']

0         1.00
1         1.00
2         1.00
3         1.00
4         1.00
          ... 
185995    3.75
185996    4.50
185997    2.50
185998    2.00
185999    4.50
Name: Valencia, Length: 186000, dtype: float64

In [ ]:
construct_cols = ["Valencia", "Emocao_Positiva", "Credibilidade"]

df_cluster_segment = (
    df_cluster_scores
    .groupby(
        [
            "cluster_id",
            "cluster_tema",
            "cluster_macrotema",
            "segmento_e_bolhas",
        ],
        as_index=False
    )[construct_cols]
    .mean()
    .round(3)
)

df_cluster_segment.to_csv(
    CLUSTER_OUTPUT_DIR / "agg_cluster_segment.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

df_cluster_segment.head()

In [45]:
df_segment_overview = (
    df_cluster_scores
    .groupby("segmento_e_bolhas", as_index=False)
    .agg(
        n_respostas=("id_persona", "count"),
        n_agentes=("id_persona", "nunique"),
        n_clusters=("cluster_id", "nunique"),
        Valencia=("Valencia", "mean"),
        Emocao_Positiva=("Emocao_Positiva", "mean"),
        Credibilidade=("Credibilidade", "mean"),
    )
)

df_segment_overview[construct_cols] = (
    df_segment_overview[construct_cols]
    .round(3)
)

df_segment_overview.sort_values(
    "Valencia",
    ascending=False
)

,segmento_e_bolhas,n_respostas,n_agentes,n_clusters,Valencia,Emocao_Positiva,Credibilidade
6,Liberais sociais,6510,7,930,3.576,4.079,5.150
3,Empreendedores individuais,9300,10,930,3.429,4.166,4.978
8,Progressistas,22320,24,930,3.388,4.083,5.148
4,Empresários,13020,14,930,3.262,4.015,5.043
7,Militantes de esquerda,18600,20,930,2.946,4.205,5.014
0,Agro,21390,23,930,2.920,3.861,4.791
1,Conservadores cristãos,48360,52,930,2.808,3.792,4.755
2,Dependentes do Estado / Classe D e E,39060,42,930,2.716,3.696,4.700
5,Extrema direita,7440,8,930,2.197,3.627,4.201


In [56]:
df_cluster_scores.columns

Index(['VAL_01', 'VAL_02', 'VAL_03', 'VAL_04', 'POS_01', 'POS_02', 'POS_03',
       'POS_04', 'POS_05', 'CRE_01', 'CRE_02', 'CRE_03', 'id_persona', 'idade',
       'gênero', 'escolaridade', 'situação ocupacional', 'estado civil',
       'condição de moradia', 'tipo de moradia', 'localização',
       'renda familiar mensal', 'acesso à internet', 'tamanho do domicílio',
       'telefone', 'identificação política', 'ideologia', 'raça/cor',
       'geração', 'segmento_e_bolhas', 'tipo_de_território', 'uf',
       'tipo_de_fontes_de_informação', 'cluster_id', 'cluster_resumo',
       'cluster_tema', 'cluster_macrotema', 'news_id', 'news_titulo',
       'news_fonte', 'news_tipo_de_fontes_de_informação', 'news_tema',
       'news_macro_tema', 'news_empresa', 'news_data', 'news_link',
       'fonte_familiar', 'model', 'error', 'Valencia', 'Emocao_Positiva',
       'Credibilidade'],
      dtype='object')

In [57]:
df_segment_macrotema = (
    df_cluster_scores
    .groupby(
        [
            "segmento_e_bolhas",
            "cluster_macrotema",
            'cluster_tema'
        ],
        as_index=False
    )
    .agg(
        n_respostas=("id_persona", "count"),
        n_agentes=("id_persona", "nunique"),
        n_clusters=("cluster_id", "nunique"),
        Valencia=("Valencia", "mean"),
        Emocao_Positiva=("Emocao_Positiva", "mean"),
        Credibilidade=("Credibilidade", "mean"),
    )
)

df_segment_macrotema[construct_cols] = (
    df_segment_macrotema[construct_cols]
    .round(3)
)

df_segment_macrotema['segmento_e_bolhas'].unique()

array(['Agro', 'Conservadores cristãos',
       'Dependentes do Estado / Classe D e E',
       'Empreendedores individuais', 'Empresários', 'Extrema direita',
       'Liberais sociais', 'Militantes de esquerda', 'Progressistas'],
      dtype=object)

In [64]:
df_segment_macrotema[df_segment_macrotema['segmento_e_bolhas'].isin(['Militantes de esquerda']) & df_segment_macrotema['cluster_macrotema'].isin(['Trabalho, renda e relações trabalhistas'])]

,segmento_e_bolhas,cluster_macrotema,cluster_tema,n_respostas,n_agentes,n_clusters,Valencia,Emocao_Positiva,Credibilidade
7081,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Acidente de trabalho em pedreira,20,20,1,1.000,3.370,5.600
7082,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Acidente de trabalho em show,20,20,1,1.000,2.990,5.567
7083,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Aprovação da PEC do trabalho,40,20,2,5.956,5.195,5.667
7084,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Aprovação da PEC sobre jornada de,20,20,1,6.100,5.150,5.667
7085,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Assédio moral no trabalho,20,20,1,3.875,4.920,5.683
7086,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Atos pelo fim da escala 6x1,20,20,1,6.062,5.270,5.900
7087,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Busca por renda alternativa no Brasil,20,20,1,2.075,4.870,5.650
7088,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Caiado e a escala de trabalho,20,20,1,5.162,4.700,5.367
7089,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Campanha contra escala de trabalho 6x1,20,20,1,6.512,5.230,6.283
7090,Militantes de esquerda,"Trabalho, renda e relações trabalhistas",Celebrações do Dia do Trabalho,20,20,1,6.125,5.270,6.017


## 18. Salvar construct scores

In [ ]:

df_scores = compute_construct_scores(df_responses)

construct_scores_path = ABERJE_OUTPUT_DIR / "construct_scores.csv"
df_scores.to_csv(construct_scores_path, index=False, sep=";", encoding="utf-8-sig")

print("Salvo em:", construct_scores_path)

df_scores[
    [
        "id_persona",
        "news_id",
        "segmento_e_bolhas",
        "geração",
        "ideologia",
        "Valencia",
        "Arousal",
        "Emocao_Positiva",
        "Emocao_Negativa",
        "Credibilidade",
        "Atratividade",
        "fonte_familiar",
        "error",
    ]
].head()


## 19. Agregações

In [ ]:

def aggregate_by_news(df_scores: pd.DataFrame) -> pd.DataFrame:
    constructs = [c for c in CONSTRUCT_MAP.keys() if c in df_scores.columns]
    return (
        df_scores
        .groupby(["news_id"], as_index=False)[constructs]
        .mean()
        .round(3)
    )


def aggregate_by_news_and_group(df_scores: pd.DataFrame, group_col: str) -> pd.DataFrame:
    constructs = [c for c in CONSTRUCT_MAP.keys() if c in df_scores.columns]
    group_cols = ["news_id", group_col]
    existing = [c for c in group_cols if c in df_scores.columns]
    return (
        df_scores
        .groupby(existing, as_index=False)[constructs]
        .mean()
        .round(3)
    )


AGG_OUTPUT_DIR = ABERJE_OUTPUT_DIR / "aggregations"
AGG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_agg_news = aggregate_by_news(df_scores)
df_agg_news.to_csv(AGG_OUTPUT_DIR / "agg_by_news.csv", index=False, sep=";", encoding="utf-8-sig")

aggregation_groups = [
    "segmento_e_bolhas",
    "geração",
    "identificação política",
    "ideologia",
    "raça/cor",
    "gênero",
    "uf",
    "tipo_de_território",
    "tipo_de_fontes_de_informação",
    "fonte_familiar",
]

aggregations = {"by_news": df_agg_news}

for group_col in aggregation_groups:
    if group_col in df_scores.columns:
        df_group = aggregate_by_news_and_group(df_scores, group_col)
        safe_name = (
            group_col
            .replace(" ", "_")
            .replace("/", "_")
            .replace("ç", "c")
            .replace("ã", "a")
            .replace("é", "e")
        )
        df_group.to_csv(
            AGG_OUTPUT_DIR / f"agg_by_news_{safe_name}.csv",
            index=False,
            sep=";",
            encoding="utf-8-sig",
        )
        aggregations[group_col] = df_group

print("Agregações geradas em:", AGG_OUTPUT_DIR)
df_agg_news.head()


## 20. Visão por segmento para apresentação

In [ ]:

if "segmento_e_bolhas" in aggregations:
    df_segment_view = aggregations["segmento_e_bolhas"]

    display_cols = [
        "news_id",
        "segmento_e_bolhas",
        "Valencia",
        "Arousal",
        "Emocao_Positiva",
        "Emocao_Negativa",
        "Credibilidade",
        "Atratividade",
    ]

    df_segment_view = df_segment_view[[c for c in display_cols if c in df_segment_view.columns]]
    df_segment_view.head(30)
else:
    print("Agregação por segmento não disponível.")


## 21. Arquivos finais

In [ ]:

print("Pasta de saída:", ABERJE_OUTPUT_DIR)
for file in sorted(ABERJE_OUTPUT_DIR.rglob("*.csv")):
    print(file)
